# AM 11B Lab Activity 1: Linear Approximation of a Production Function

**Week 1 — built on: Functions of Several Variables, Partial Derivatives, Linear Approximation**

**Authors:** Julie Simons, with Claude (Sonnet 5, Anthropic), via Claude Code — September 2026

*With special thanks to Mariana Bessa and prior AM 11B students, whose earlier notebooks inspired this material, and to Eric Van Dusen and the [Data 88E](https://github.com/data-88e) course for inspiration.*

![Workers and robotic arms building a car on an assembly line](https://thumb.wikimedia.org/wikipedia/commons/thumb/f/f1/Hyundai_car_assembly_line.jpg/960px-Hyundai_car_assembly_line.jpg)

*Photo: Anonyme, CC BY 2.5, via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Hyundai_car_assembly_line.jpg)*

<!-- canvas-group-instructions -->
## Group Submission Instructions

**Joining your lab group on Canvas**

1. Agree with your partners on a group number (e.g., "Lab Group 7").
2. In our Canvas course, click **People**, then the **Groups** tab.
3. Find your group and click **Join**. **Every member must do this themselves.** You can't add your partners.
4. To check, click the group name and make sure all your partners are listed. If you joined the wrong group, click **Leave** and join the right one.
5. When you submit, **only one person uploads the notebook**. It counts for everyone in the group.

Please join your group before submitting your first lab. If you submit before joining, only you will get credit.

Please write the name(s) of your group (or yourself) below by double-clicking on this text.

**NAME(S):** *Replace this text with your name(s)*

## New to Jupyter Notebooks? Start here.

A notebook is a sequence of **cells**. There are two kinds you'll see today:

- **Markdown cells** (like this one) hold text, explanations, and questions. Double-click one to edit it, then run it (see below) to render the formatted text again.
- **Code cells** (gray background, with `[ ]` to the left) hold Python code that actually computes something.

**To run a cell:** click on it, then press **Shift + Enter**. This runs the cell and moves to the next one. You'll do this a lot today.

**A few things that trip people up the first time:**
- Cells must generally be run **in order, top to bottom**, the first time through, because later cells often use variables defined in earlier ones. If you get a `NameError`, it usually means you skipped a cell above.
- If things get weird (variables seem stuck at old values), use the menu to **Restart the kernel and run all cells** — this is the "turn it off and on again" of notebooks, and it's completely normal to need it.
- Editing a code cell where it says `# EDIT HERE` and re-running it (Shift+Enter) is how you'll interact with most of today's activities. You don't need to know much Python beyond changing a number or a short formula.
- Nothing you do here can break anything permanently — feel free to experiment. Worst case, re-download a fresh copy from Canvas.

That's genuinely all the Jupyter knowledge you need for today. Let's get started.

## Learning Goals

By the end of this lab you should be able to:
1. Recall what a marginal function of one variable represents, as a warm-up.
2. Compute partial derivatives of a two-input production function by hand and interpret them as marginal products.
3. Build a **linear approximation** (tangent plane) to a multivariable function at a point, and evaluate it by hand for a chosen input change.
4. Use the linear approximation to estimate how output changes for a small change in inputs — without recomputing the full function.
5. See, numerically and visually, *why* linear approximation works well close to the anchor point and gets worse far away.

### Why economists care
A firm rarely wants to recompute its entire production function from scratch every time labor or capital shifts a little. Linear approximation — built from marginal products — gives a fast, "good enough near home" estimate of how output responds to small input changes. This is the same idea behind concepts like marginal cost pricing and short-run forecasting.

## Setup

Run the cell below to load the tools we'll use today.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import sympy as sp
import random

sp.init_printing()
plt.rcParams['figure.figsize'] = (9, 6)
%matplotlib inline

def checkpoint(is_correct, hint=""):
    if is_correct:
        emoji = random.choice(["✅", "🚀", "⭐"])
        msg = random.choice([
            "Great job! You nailed it.",
            "You did it!",
            "Nailed it — nice work.",
            "Yes! That's exactly right.",
            "Correct! You're crushing this.",
        ])
        print(f"{emoji} {msg}")
    else:
        print("❌ Try again!" + (f" {hint}" if hint else ""))

# Answer checks live in lab_checks.py (in the same folder as this notebook)
try:
    from lab_checks import check, same_formula
except ImportError:
    def check(*args):
        print("ℹ️ Answer checking isn't available here because lab_checks.py isn't in the same folder "
              "as this notebook. Your answers are still saved.")

print("Setup complete!")

## Quick Review: Marginal Functions of One Variable

Before working with functions of two inputs, let's remind ourselves what a **marginal function** means for a function of a single variable — you've seen this idea in a previous course, or earlier in this one.

If $f(x)$ is some economic quantity (total cost, total revenue, total utility, ...) written as a function of a single input $x$, then its **marginal function** is the derivative $f'(x)$. It represents the *rate of change* of $f$ at $x$ — approximately how much $f$ changes if $x$ increases by one more small unit, holding everything else fixed.

**Example:** if $C(x)$ is the total cost of producing $x$ units, then $C'(x)$ is the *marginal cost* — the approximate cost of producing one additional unit, given that you're already producing $x$ units.

Run the cell below to see this with a concrete cost function.

In [ ]:
x = sp.symbols('x')
C = x**2 + 10*x + 50   # total cost ($) of producing x units

C_prime = sp.diff(C, x)
print("Total cost:    C(x)  =", C)
print("Marginal cost: C'(x) =", C_prime)

x0 = 20
print(f"\nMarginal cost at x={x0}: C'({x0}) = {C_prime.subs(x, x0)}")
print("Interpretation: producing the 21st unit costs approximately that many additional dollars.")

### Warm-Up Question: Marginal Revenue

A firm's total revenue from selling $x$ units is:

$$R(x) = -2x^2 + 200x$$

**a)** In the code cell below, use `sp.diff` (just like the example above) to find the marginal revenue function $R'(x)$, and evaluate it at $x=40$ and at $x=60$.

**b)** Then, in the markdown cell below that, interpret what the number $R'(60)$ means in words, and explain why it makes sense that marginal revenue is *smaller* at $x=60$ than at $x=40$ for this particular revenue function.

In [ ]:
x = sp.symbols('x')
R = -2*x**2 + 200*x   # total revenue ($) from selling x units

R_prime = None   # EDIT HERE: replace None with sp.diff(R, x)

print("Marginal revenue function R'(x) =", R_prime)
print("R'(40) =", R_prime.subs(x, 40))
print("R'(60) =", R_prime.subs(x, 60))

# Check your answer (the checking code lives in lab_checks.py)
check("lab1_warmup", R_prime)

*Replace this text with your interpretation of $R'(60)$, and your explanation for why marginal revenue is smaller at $x=60$ than at $x=40$.*

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

## The Scenario

Now let's extend this same idea — a rate of change telling you the approximate effect of a one-unit increase — to a function of *two* inputs.

A firm's output (in units) is given by the production function

$$Q(L, K) = 20\sqrt{LK}$$

where $L$ is labor (worker-hours) and $K$ is capital (machine-hours). The firm is currently operating at $(L_0, K_0) = (100, 25)$.

### Step 1: Find the marginal products at the current input levels — try it by hand first

Recall that the **marginal product of labor** is $MP_L = \frac{\partial Q}{\partial L}$ and the **marginal product of capital** is $MP_K = \frac{\partial Q}{\partial K}$.

**Before running any code**, grab a pencil and paper:

1. Rewrite $Q(L,K) = 20\sqrt{LK}$ as $Q = 20 L^{1/2} K^{1/2}$, and use the power rule (treating the other variable as a constant, just like in the warm-up above) to find the *formulas* for $MP_L$ and $MP_K$.
2. Since $L_0=100$ and $K_0=25$ are both perfect squares ($\sqrt{100}=10$, $\sqrt{25}=5$), plug in and simplify by hand to get exact numbers for $Q_0$, $MP_L$, and $MP_K$ at $(100,25)$ — no calculator needed.

Type your three numbers into the code cell below and run it.

</div>

In [ ]:
# EDIT HERE: replace each None with the value you computed by hand
Q0_hand  = None   # output Q at (100, 25)
MPL_hand = None   # marginal product of labor at (100, 25)
MPK_hand = None   # marginal product of capital at (100, 25)

Now we'll verify your answers above. The cell below has Python compute $MP_L$ and $MP_K$ by taking the partial derivatives, evaluate everything at $(L_0, K_0) = (100, 25)$, and compare the results with the values you entered.

If the checkpoint says to try again, look back at your hand work for the value it names, fix your number in the cell above, and rerun both cells.

In [ ]:
L, K = sp.symbols('L K', positive=True)
Q = 20 * sp.sqrt(L*K)

MPL = sp.diff(Q, L)
MPK = sp.diff(Q, K)

print("Q(L,K)  =", Q)
print("MP_L    =", sp.simplify(MPL))
print("MP_K    =", sp.simplify(MPK))

L0, K0 = 100, 25
Q0   = float(Q.subs({L: L0, K: K0}))
MPL0 = float(MPL.subs({L: L0, K: K0}))
MPK0 = float(MPK.subs({L: L0, K: K0}))

print(f"\nAt (L0,K0)=({L0},{K0}):")
print(f"  Q0   = {Q0:.3f} units")
print(f"  MP_L = {MPL0:.4f}  (extra output from one more worker-hour)")
print(f"  MP_K = {MPK0:.4f}  (extra output from one more machine-hour)")
print()

# Check your answer (the checking code lives in lab_checks.py)
check("lab1_step1", Q0_hand, MPL_hand, MPK_hand)

### A Broader Pattern: Diminishing Marginal Returns

The marginal product you just computed, $MP_K=20$, is only the rate of change of output *right at* $(L_0,K_0)=(100,25)$ — and that rate itself changes as capital changes. Run the cell below to see $MP_K$ plotted as a function of $K$, holding labor fixed at $L_0=100$. Notice it's *decreasing*: each additional machine-hour adds less extra output than the one before it — a pattern called **diminishing marginal returns**.

This is the same underlying reason a linear approximation — which freezes the marginal rate at a single point — drifts away from the truth as you move farther from that point, which you'll see directly in Question 1.2 below.

In [ ]:
K_range = np.linspace(5, 200, 100)
MPK_of_K = sp.lambdify(K, MPK.subs(L, L0), 'numpy')
MPK_vals = MPK_of_K(K_range)

plt.figure(figsize=(8, 5))
plt.plot(K_range, MPK_vals, 'b-', linewidth=2)
plt.axvline(K0, color='gray', linestyle=':', label=f'Current capital $K_0={K0}$')
plt.xlabel('Capital (K), with Labor fixed at $L_0=100$')
plt.ylabel('Marginal Product of Capital, $MP_K$')
plt.title('Diminishing Marginal Returns to Capital')
plt.legend()
plt.show()

<div style="background-color:#fff6dd; border-left:6px solid #e0ac00; border-radius:4px; padding:4px 18px 4px 18px; color:#5c4400; margin:12px 0;">

### Step 2: Build the linear approximation — try it by hand first

The linear approximation (tangent plane) to $Q$ at $(L_0, K_0)$ is:

$$Q(L,K) \approx Q(L_0,K_0) + MP_L(L_0,K_0)\,(L - L_0) + MP_K(L_0,K_0)\,(K - K_0)$$

This is just the multivariable version of "new value $\approx$ old value + (rate of change) $\times$ (how much things changed)."

**Before running any code**, use the $Q_0=1000$, $MP_L=5$, $MP_K=20$ you just found (by hand or checked with the cell above) to estimate $Q(105, 27)$ by hand — it's just multiplication and addition, no square roots required. Type your estimate into the code cell below and run it.

</div>

In [ ]:
approx_hand = None   # EDIT HERE: replace None with your hand-computed estimate of Q(105, 27)

Now we'll verify your estimate above. The cell below evaluates the same approximation formula in code to check your arithmetic, and — since $105 \times 27$ isn't a perfect square — uses Python to get the *true* value of $Q(105,27)$, which isn't something you'd want to compute by hand.

In [ ]:
def linear_approx(L1, K1):
    return Q0 + MPL0*(L1 - L0) + MPK0*(K1 - K0)

L1, K1 = 105, 27
approx = linear_approx(L1, K1)
actual = float(Q.subs({L: L1, K: K1}))
pct_error = abs(actual - approx) / actual * 100

print(f"Linear approximation at (L,K)=({L1},{K1}): {approx:.3f}")
print(f"Actual Q(L,K):                          {actual:.3f}")
print(f"Percent error:                           {pct_error:.4f}%")

print()

# Check your answer (the checking code lives in lab_checks.py)
check("lab1_step2", approx_hand)

### Question 1.1: Interpreting the Approximation

The approximation and the actual value are extremely close here. In one or two sentences, explain *why* it makes sense to call $(105, 27)$ a "nearby" point relative to $(100, 25)$, and why that matters for how accurate a linear approximation is.

*Replace this text with your answer.*

## Visualizing the Tangent Plane

Run the cell below to see the production surface $Q(L,K)$ (blue-green) together with the tangent plane at $(L_0, K_0)$ (semi-transparent). Notice how closely the plane hugs the surface right around the anchor point — and how it drifts away as you move farther out.

In [ ]:
L_vals = np.linspace(10, 250, 40)
K_vals = np.linspace(5, 80, 40)
L_grid, K_grid = np.meshgrid(L_vals, K_vals)
Q_grid = 20 * np.sqrt(L_grid*K_grid)
plane_grid = Q0 + MPL0*(L_grid - L0) + MPK0*(K_grid - K0)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(L_grid, K_grid, Q_grid, cmap='viridis', alpha=0.75)
ax.plot_surface(L_grid, K_grid, plane_grid, color='red', alpha=0.25)
ax.scatter([L0], [K0], [Q0], color='black', s=60, label='Anchor point $(L_0,K_0)$')
ax.set_xlabel('Labor (L)')
ax.set_ylabel('Capital (K)')
ax.set_zlabel('Output (Q)')
ax.set_title('Production Surface (blue-green curved surface) and Tangent Plane (red)')
ax.legend()
plt.show()

<div style="background-color:#e7f1fc; border-left:6px solid #4a90d9; border-radius:4px; padding:4px 18px 4px 18px; color:#0b3d78; margin:12px 0;">

## Your Turn

Below, the code recomputes the linear approximation and the actual output at a new point you choose — and reports how big the error is. Try a point that's **much farther** from $(L_0,K_0)=(100,25)$ than the example above, for instance somewhere around $(200, 10)$ or $(60, 40)$.

Before you edit the cell: jot down what the approximation *formula* predicts for the point you're about to try — it's the same quick arithmetic as Step 2 above, just with your own numbers. Then edit `L1_new` and `K1_new` in the cell marked `# EDIT HERE` and run it to see how far off that estimate actually is.

</div>

In [ ]:
L1_new = None   # EDIT HERE -- pick a labor value far from L0=100
K1_new = None   # EDIT HERE -- pick a capital value far from K0=25
assert L1_new is not None and K1_new is not None, "Set L1_new and K1_new to numbers before running this cell!"

approx_new = linear_approx(L1_new, K1_new)
actual_new = float(Q.subs({L: L1_new, K: K1_new}))
pct_error_new = abs(actual_new - approx_new) / actual_new * 100

print(f"Linear approximation at (L,K)=({L1_new},{K1_new}): {approx_new:.3f}")
print(f"Actual Q(L,K):                                {actual_new:.3f}")
print(f"Percent error:                                  {pct_error_new:.4f}%")

# Check your answer (the checking code lives in lab_checks.py)
check("lab1_your_turn", pct_error_new, pct_error)

### Question 1.2: Why does the error grow?

Compare the percent error you just found to the roughly 0.01% error from the $(105,27)$ example. In your own words, explain *why* a linear approximation — which is built entirely from information at a single point $(L_0,K_0)$ — necessarily gets worse the farther away you evaluate it. (Hint: think about what a tangent plane assumes about the surface's curvature.)

*Replace this text with your answer.*

## Reflection

**In 1-2 sentences:** Can you think of a real business situation where a linear approximation like this would be *good enough* to use, and one where it would be too risky to rely on? Consider how "small" a change has to be for this shortcut to be trustworthy.

*Replace this text with your answer.*

---

## Submission

1. Make sure you've run all cells in order (**Runtime/Kernel → Restart and run all** is a good final check).
2. Answer the short reflection question(s) above by editing the markdown cells.
3. Go to **File → Download** to save the `.ipynb` file to your computer.
4. Upload the file to Canvas under this Lab Activity.

Only one person per group needs to submit if you worked together — just make sure everyone's name is listed at the top.

**This lab is designed to take about 30 minutes.** If you're stuck for more than a few minutes on any one cell, that's your cue to ask a neighbor, a TA, or post on Ed Discussion — don't burn your whole discussion period on one line of code.
